### Setup

In [ ]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import torch
import random
import numpy as np
from datetime import datetime
import json
import shutil
from datasets import Dataset, DatasetDict
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from huggingface_hub import login
from diffusers import DDPMScheduler, StableDiffusionPipeline, StableDiffusionPipeline
from metric import *

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ', device)   

# CONFIG
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")
MASTER_SEED = 42
IMG_SIZE = 512
BATCH_SIZE = 4
TRAIN_DATA_SIZE = 2024
TEST_DATA_SIZE = 506

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
## DATA
TEST_IMAGE_FOLDER = config.get("TEST_IMAGE_FOLDER")   
TEST_LABEL_FOLDER = config.get("TEST_LABEL_FOLDER") 
TEST_IMAGE_FILE = sorted([file for file in os.listdir(TEST_IMAGE_FOLDER) if file.endswith(('.jpg', '.jpeg', '.png'))], key=lambda x: int(x.split('.')[0]))[:TEST_DATA_SIZE]
TEST_LABEL_FILE = sorted([file for file in os.listdir(TEST_LABEL_FOLDER) if file.endswith(('.json'))], key=lambda x: int(x.split('.')[0]))[:TEST_DATA_SIZE]

## DATA for nike, zara
# TEST_IMAGE_FOLDER = config.get("NIKE_IMAGE_FOLDER") 
# TEST_LABEL_FOLDER = config.get("NIKE_LABEL_FOLDER") 
# TEST_IMAGE_FOLDER = config.get("ZARA_IMAGE_FOLDER") 
# TEST_LABEL_FOLDER = config.get("ZARA_LABEL_FOLDER") 
# TEST_IMAGE_FILE = sorted([file for file in os.listdir(TEST_IMAGE_FOLDER) if file.endswith(('.jpg', '.jpeg', '.png'))], key=lambda x: int(x.split('.')[0][4:]))[TRAIN_DATA_SIZE:]
# TEST_LABEL_FILE = sorted([file for file in os.listdir(TEST_LABEL_FOLDER) if file.endswith(('.json'))], key=lambda x: int(x.split('.')[0][4:]))[TRAIN_DATA_SIZE:]

## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250604_234758'

# HUGGINGFACE
HUGGING_FACE_TOKEN = config.get("HUGGING_FACE_TOKEN")
login(HUGGING_FACE_TOKEN)

2025-06-10 14:35:33.126364: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-10 14:35:33.166712: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-10 14:35:33.744217: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Instructions for updating:
non-resource variables are not supported in the long term
device :  cuda
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/gayeon39/.cache/huggingface/token
Login successful


### Load Model

In [ ]:
# load orginal model 
pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16).to(device)
# load fine-tunined model 
pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True)

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(torch.float16).to(device)  
vae = pipe.vae.to(torch.float16).to(device)  
unet = pipe.unet.to(torch.float16).to(device)  

noise_scheduler = DDPMScheduler.from_pretrained(
    PRE_TRAINED_MODEL_NAME,
    subfolder="scheduler"
)
if hasattr(noise_scheduler, "alphas_cumprod"):
    noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

### Preprocess Data

In [3]:
# Function to tokenize the text column in the given data sample and return token ID tensors.
def tokenize_captions(examples, caption_column='text', is_train=True):
    captions = []
    for caption in examples[caption_column]:
        # If there is only one caption
        if isinstance(caption, str):
            captions.append(caption)
        # If there are multiple captions, randomly select one
        elif isinstance(caption, (list, np.ndarray)):
            captions.append(random.choice(caption) if is_train else caption[0])  # Take a random caption if training, otherwise take the first one
        else:
            raise ValueError(
                f"Caption column `{caption_column}` should contain either strings or lists of strings."
            )
    
    inputs = tokenizer(
        captions, max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt"
    )

    return inputs.input_ids

# Transformation pipeline for preprocessing image data for training
transforms = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(IMG_SIZE) if True else transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip() if True else transforms.Lambda(lambda x: x),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),  # Convert (0,1) -> (-1,1)
    ]
)

# Function to preprocess images by converting them to RGB, applying transformations, 
# and tokenizing the text column to construct a training dataset.
def preprocess_data(examples, image_column='image'):
    images = [image.convert("RGB") for image in examples[image_column]]
    # Image preprocessing
    examples["pixel_values"] = [transforms(image) for image in images]
    # Text preprocessing
    examples["input_ids"] = tokenize_captions(examples)
    return examples

# Function to batch images and token IDs, stacking them into PyTorch tensor format.
def collate_fn(examples):
    # (C, H, W), ..., (C, H, W) -> stack -> (N, C, H, W): Stack along N dimension
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    # Same as tensor.contiguous()
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()
    # {(77,), ..., (77,)}_N samples -> stack -> (N, 77)
    input_ids = torch.stack([example["input_ids"] for example in examples])

    return {"pixel_values": pixel_values, "input_ids": input_ids}

In [ ]:
data = []
for image_file, label_file in zip(TEST_IMAGE_FILE, TEST_LABEL_FILE):
    image_path = os.path.join(TEST_IMAGE_FOLDER, image_file)
    label_path = os.path.join(TEST_LABEL_FOLDER, label_file)
    with open(image_path, 'rb') as image:
        image_data = Image.open(image)
        image_data = image_data.convert('RGB')
    with open(label_path, 'r') as label:
        label_data = json.load(label)
    data.append({
        'image':image_data,
        'text':label_data
    })
    
# Converting to Dataset
if TEST_IMAGE_FOLDER[23:] == "Nike/" or TEST_IMAGE_FOLDER[23:] == "Zara/":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text']['Summary'] for item in data]
    })
else:
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text']['Summary'] for item in data]
    })
# Create Test DatasetDict and Apply Preprocessing
test_dataset = DatasetDict({'test': dataset})
test_dataset = test_dataset.with_transform(preprocess_data)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset['test'],
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=BATCH_SIZE
)
print(test_dataset)

### Test Model

In [ ]:
def calculate_fid(test_dataset, test_label_file, pipe):
    from torchvision import transforms as T
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fid_metric = FrechetInceptionDistance(feature=64).to(device)
    fid_metric.reset()
    
    # Use transform to convert PIL images to uint8 tensors
    transform = T.PILToTensor()
    
    # Create a temporary folder (for storing created images)
    tmp_folder = '../Data/Total/tmp'
    os.makedirs(tmp_folder, exist_ok=True)
    for idx, filename in tqdm(enumerate(test_label_file)):
        prompt = test_dataset['test'][idx]['text'][16:]
        generated_image = pipe(prompt).images[0]
        # Save generate image   
        filename = filename.split('.')[0]
        generated_image.save(f'{tmp_folder}/{filename}.jpg')
    
        # real image
        real_image = test_dataset['test'][idx]['image']
        
        fake_tensor = transform(generated_image).unsqueeze(0).to(device)
        real_tensor = transform(real_image).unsqueeze(0).to(device)
        
        fid_metric.update(real_tensor, real=True)
        fid_metric.update(fake_tensor, real=False)
    
    fid_score = fid_metric.compute().item()
    return fid_score

fid_score = calculate_fid(test_dataset, TEST_LABEL_FILE, pipe)

In [ ]:
# Compute precision and recall by comparing generated images with original images
GENERATE_IMAGE_FOLDER = '../Data/Total/tmp'
# data : train
GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0]))
# data : nike or zara
# GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0][4:]))

# Initialize TensorFlow session and evaluator
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)
evaluator = Evaluator(sess)
evaluator.warmup()

# Prepare batch generator (original images)
seed_batches = batch_generator(TEST_IMAGE_FILE, TEST_IMAGE_FOLDER, batch_size=64)
# Prepare batch generator (generated images)
augment_batches = batch_generator(GENERATE_IMAGE_FILE, GENERATE_IMAGE_FOLDER, batch_size=64)

# Extract activations for original images (pool_3 features)
seed_activations = evaluator.compute_activations(seed_batches) 
augment_activations = evaluator.compute_activations(augment_batches)

# Compute precision and recall
precision, recall = evaluator.compute_prec_recall(seed_activations[0], augment_activations[0])
f1 = 2 * precision * recall / (precision + recall) 
# Remove folder if it exists
shutil.rmtree(GENERATE_IMAGE_FOLDER)

print(f'FID Score: {fid_score}, Precision: {precision}, Recall: {recall}, F1: {f1}')